_We'll build this together, live. This session's main exercises covered Dijkstra's algorithm on weighted graphs; this application applies it to real airfare data._

## Application: Cheapest Multi-Leg Flight Routes

`data/airfare_routes.csv` in this folder is real data: the US Bureau of Transportation Statistics runs the **DB1B Origin and Destination Survey**, a real 10% sample of all domestic airline tickets sold each quarter. From the Q1 2025 survey, this file has the **passenger-weighted average nonstop fare** for every real direct route between 40 US airports (20 major hubs like ATL, ORD, DFW, plus 20 smaller regional airports like Boise and Providence) — 897 real routes, each backed by hundreds or thousands of real ticket sales.

Two things worth noticing about this graph before we use it:
- It's **directed**: the fare from A to B is not necessarily the same as B to A (real airline pricing is asymmetric — different demand each direction).
- Most **regional** airports have no direct flight to each other at all — you can only get from one to the other via one or more major-hub connections, and *which* hub is cheapest is rarely obvious.

**The question**: what's the cheapest total-cost way to fly from airport A to airport B, possibly through one or more connections? That's exactly Dijkstra's algorithm — same mechanism you just implemented, just tracing dollars instead of an abstract "cost".

### A note on tools

This case study runs Dijkstra's algorithm on a real weighted network, the same idea as your class-9 exercise. But instead of importing the `Digraph`/`dijkstra` code you wrote there, we'll use **`networkx`**, Python's standard graph library.

Why not reuse your own code? This case study is meant to apply the *idea* of Dijkstra's algorithm to a new, real problem, not to test whether your class-9 implementation is correct, or depend on you having finished it yet. Using `networkx` also reflects how Dijkstra's algorithm actually gets used outside this course: `networkx` is the standard graph library in Python, so this is a chance to see the same algorithm through a professional tool instead of hand-rolled code.

Two functions cover everything you need here:

- `networkx.DiGraph()`: creates a directed graph, the same idea as the `Digraph` class from class 9. Add weighted edges with `.add_edge(u, v, weight=w)`.
- `networkx.single_source_dijkstra(G, start)`: runs Dijkstra's algorithm from `start` and returns `(dists, paths)` -- a `{node: cost}` dictionary of cheapest cost to every reachable node, and a `{node: [route]}` dictionary of the actual cheapest route to each one, exactly like the `dijkstra(graph, start)` function from class 9 (which returned `dists` and `prev_nodes` instead).

```python
import networkx as nx

G = nx.DiGraph()
G.add_edge('a', 'b', weight=5)

dists, paths = nx.single_source_dijkstra(G, 'a')
```

## Part 1: Tracing cost from one airport

**Step 1: build the graph.** Read `data/airfare_routes.csv` and add a directed, weighted edge from each origin to each destination, using `networkx.DiGraph`.

**Hint:** `csv.DictReader` reads each row of a CSV file as a dictionary, keyed by the column names in the header row. Here the header includes `origin,dest,avg_fare`, so each row becomes `{'origin': ..., 'dest': ..., 'avg_fare': ..., ...}`:

In [1]:
import csv

with open('data/airfare_routes.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)
        break  # just show the first row

{'origin': 'ATL', 'dest': 'AVL', 'avg_fare': '285.76', 'sample_passengers': '106', 'n_records': '94'}


In [11]:
import csv
import networkx as nx

def build_airfare_graph(filename):
    """Builds a directed, weighted graph of real nonstop airfares."""
    g = nx.DiGraph()
    with open('data/airfare_routes.csv') as f:
        reader = csv.DictReader(f)
        for row in reader:
            g.add_edge(row['origin'],row['dest'], weight = row['avg_fare'])
    return g

airfare_graph = build_airfare_graph('data/airfare_routes.csv')

**Explore what you got back.** Whenever a package function returns something new, it's worth pausing to look at it directly -- its type, a sample of its contents -- before assuming its shape from memory. That's the normal way of working with any library, not just `networkx`.

In [28]:
print(type(airfare_graph))
print(list(airfare_graph.nodes)[:5])
print(list(airfare_graph.edges(data=True))[:3])

<class 'networkx.classes.digraph.DiGraph'>
['ATL', 'AVL', 'BOI', 'BOS', 'BTV']
[('ATL', 'AVL', {'weight': 285.76}), ('ATL', 'BOI', {'weight': 412.43}), ('ATL', 'BOS', {'weight': 212.76})]


**Step 2: run Dijkstra.** Use `networkx.single_source_dijkstra(graph, start)` -- it returns `(dists, paths)`: the cheapest cost to reach every node from `start`, and the actual cheapest route to each one, exactly like the `dijkstra` function from class 9. Let's use it here, starting from Boise (`'BOI'`).

In [65]:
# your code here — run dijkstra from 'BOI'
import csv
import networkx as nx

def build_airfare_graph(filename):
    """Builds a directed, weighted graph of real nonstop airfares."""
    g = nx.DiGraph()
    with open(filename) as f:
        reader = csv.DictReader(f)
        for row in reader:
            g.add_edge(row['origin'],row['dest'], weight = float(row['avg_fare']))
    return g

airfare_graph = build_airfare_graph('data/airfare_routes.csv')
dists, paths = nx.single_source_dijkstra(airfare_graph, 'BOI')

**Same idea here:** look at what `dists` and `paths` actually are before using them.

In [30]:
print(type(dists), type(paths))
print(list(dists.items())[:5])
print(paths['PVD'])

<class 'dict'> <class 'dict'>
[('BOI', 0), ('LAS', 158.95), ('SEA', 159.77), ('SFO', 197.79), ('PHX', 198.93)]
['BOI', 'MCO', 'GSP', 'PVD']


**Step 3: read off a cost, and the actual route.** `dists['PVD']` gives the cheapest total price from Boise to Providence. `paths['PVD']` gives the actual sequence of airports as a list -- which may well go through a connection nobody would guess by eye.

In [ ]:
target = 'PVD'

distance = 0

for i in range(len(paths['PVD'])-1):
    distance += airfare_graph.get_edge_data(paths[i],paths[i+1])['weight']
    
return distance 


KeyError: 0

## Part 2: Your turn

Using `airfare_graph`, `dists`, and `paths`, answer:

1. Which destination is **most expensive** to reach from `'BOI'` (largest value in `dists`)? What's the cheapest route there, and how many stops does it take?
2. Does Dijkstra from `'BOI'` reach **every** other airport in the graph, or are some unreachable? (Unreached nodes simply won't appear as keys in `dists`.)

(Hint: `airfare_graph.nodes` gives you every airport in the graph.)

In [92]:
# print(airfare_graph.nodes)
# dists, paths = nx.single_source_dijkstra(airfare_graph, 'BOI')
# print(max(dists))
# paths['XNA']

most_expensive_airport = max(dists, key = dists.get)
print(most_expensive_airport)
dists[most_expensive_airport]

# for key in dists:
#     if key not in airfare_graph.nodes:
#         print(key)

len(dists) == len(airfare_graph.nodes())

MIA


True

## Part 3: Extension — where can you go for under $250?

Given a starting airport and a budget, which destinations can you actually afford? Run Dijkstra once from `'BOI'`, then for each of the budgets $250 and $400, count how many airports are reachable with total cost at most that budget.

In [ ]:
dists, _ = nx.single_source_dijkstra(airfare_graph, 'BOI')

for budget in (250, 400):
    # your code here — count airports reachable from BOI with cost at most `budget`, and print it
    pass

## Part 4: Extension — the cheapest airport to fly out of

So far we've only measured cost *from* `'BOI'`. Which airport, if you were based there, would give the cheapest cost to reach everywhere else?

For every airport in the graph, run Dijkstra from it and compute the *average* cost to every other reachable airport (use the average, not the total — an airport that only reaches a handful of destinations would look unfairly cheap on a total, even if each of its routes is expensive). Then find the airport with the lowest average — the cheapest airport to be based at.

In [ ]:
averages = {}
for airport in airfare_graph.nodes:
    # your code here — run Dijkstra from `airport`, compute the average cost to every other reachable airport, store it in `averages[airport]`
    pass

# your code here — find and print the airport with the lowest average cost

## Part 5: Extension — how asymmetric are fares really?

The intro mentioned that fares are directional: the cost from A to B isn't necessarily the cost from B to A. Now that you can run Dijkstra from any airport, you can measure this directly.

Run Dijkstra from every airport and keep each result. Then, for every pair of airports `(a, b)` that are reachable from each other in both directions, compare the cost `a -> b` to the cost `b -> a`, and count:

- how many such pairs have (exactly) equal cost in both directions vs different cost,
- and, among the ones with different cost, the minimum, maximum, and average absolute difference.

(Hint: round differences to the nearest cent with `round(..., 2)` before comparing, since tiny floating-point noise can otherwise make two equal-looking fares look different.)

In [ ]:
all_dists = {}
for start in airfare_graph.nodes:
    # your code here — run Dijkstra from `start`, store the result in `all_dists[start]`
    pass

# your code here — loop over every pair (a, b), compare cost a->b to cost b->a,
# then print the equal-cost count, asymmetric count, and the min/max/average asymmetric difference